In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import json
import torch
import matplotlib.pyplot as plt

from utils import *
from plot import *

In [2]:
# !python dataset.py --dataset multif --ratio 0 --multi_turn

filter dataset

In [3]:
instruction_list = [
    "detectable_content:number_placeholders",
    "detectable_content:postscript",
    "detectable_format:json_format",
    "detectable_format:multiple_sections",
    "detectable_format:number_bullet_lists",
    "detectable_format:number_highlighted_sections",
    "detectable_format:title"
]

turn = 0


In [4]:
base_f = './multif'
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
sae_id = './llama_scope/Llama3_1-8B-Base-L6R-8x'


In [5]:
generate = Generate(model_id, sae_id, base_f, device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 120.44it/s]


In [6]:
generate.load_multif_data(data_f='multif.json', 
                          filter_kwargs={
                              # "instruction_list": instruction_list, 
                              "turn": turn
                              })


In [7]:
generate.subset[6 : 8]

[(6,
  '1019:15:en_0',
  {'messages': [{'role': 'user',
     'content': 'Given the sentence "Two young boys with toy guns and horns." can you ask a question? Please ensure that your response is in English, and in all lowercase letters. No capital letters are allowed.'}],
   'reference': {'instruction_id_list': ['change_case:english_lowercase'],
    'kwargs': ['{}']},
   'metric': 'rule_based',
   'language': 'English'}),
 (7,
  '1019:15:en_1',
  {'messages': [{'role': 'user',
     'content': 'Given the sentence "Two young boys with toy guns and horns." can you ask a question? Please ensure that your response is in English, and in all lowercase letters. No capital letters are allowed.'},
    {'role': 'user',
     'content': 'The result must contain a title wrapped in double angular brackets, i.e. <<title>>.'}],
   'reference': {'instruction_id_list': ['change_case:english_lowercase',
     'detectable_format:title'],
    'kwargs': ['{}', '{}']},
   'metric': 'rule_based',
   'language': 

In [8]:
# pred = generate.run(layer=6,
#                     save_results=True,
#                     stitch_turns=True,
#                     mode="both",
#                     act_out_f="activations_topk")


In [9]:
ids = generate.load_multif_results(data_f='predictions.json')
                                #    instruction_list=instruction_list

In [10]:
# ids

In [11]:
counts = {
    inst: {
        turn: {
            k: len(v[k]) for k in ["pass", "fail"]
        }
        for turn, v in turns.items()
    }
    for inst, turns in ids.items()
}

# counts

In [12]:
source = 'user'
selector = 'all'
pooling = 'mean'


ids_f = collate_ids_by_verdict_turn(ids, verdict_turn=2)

instances = get_activation_instances(ids_f, 
                                     base_f=base_f, 
                                     source=source, 
                                     selector=selector, 
                                     pooling=pooling, 
                                     lexical_only=False,
                                     all_turns=True)


In [13]:
instances.keys()

dict_keys(['pass', 'fail', 'all'])

In [14]:
instances_stats = {}

for i in ['all', 'pass', 'fail']:
    instances_stats[i] = get_stats(instances[i], level='turn')

In [15]:
instances_stats['pass']

{'level': 'turn',
 'stats': {1: {'count': 625,
   'mean': tensor([8.1674e-05, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
           0.0000e+00]),
   'median': tensor([0., 0., 0.,  ..., 0., 0., 0.]),
   'std': tensor([0.0010, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]),
   'var': tensor([1.0567e-06, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
           0.0000e+00]),
   'mean_l0': 605.23681640625,
   'std_l0': 63.25745391845703,
   'l0_distribution': [670.0,
    549.0,
    573.0,
    654.0,
    625.0,
    520.0,
    625.0,
    555.0,
    563.0,
    642.0,
    649.0,
    647.0,
    618.0,
    690.0,
    571.0,
    561.0,
    546.0,
    723.0,
    664.0,
    576.0,
    580.0,
    621.0,
    621.0,
    636.0,
    718.0,
    668.0,
    538.0,
    583.0,
    679.0,
    606.0,
    695.0,
    487.0,
    684.0,
    615.0,
    664.0,
    583.0,
    586.0,
    583.0,
    690.0,
    501.0,
    723.0,
    671.0,
    725.0,
    549.0,
    571.0,
    688.0,
    645.0,
    479.

In [16]:
plot_l0(instances_stats['all'])

alt.FacetChart(...)

In [17]:
plot_l0(instances_stats['pass'])

alt.FacetChart(...)

In [18]:
plot_l0(instances_stats['fail'])

alt.FacetChart(...)

In [4]:
from multif.analyze import DegradationAnalysis


# load
da = DegradationAnalysis(
    predictions_path="multif/predictions.json",
    activations_dir="multif/activations_topk",
)
da.load()
da.save_summary()

Loaded 909 conversations, 1355 instruction trajectories.
  pass_all             777
  fail_all             411
  interference         72
  forgetting           53
  mixed                42


'multif/plots/summary.txt'

In [6]:
from multif.plot import *


i = '1005:3:en'
n_tokens = 25
k_features = 50

# token activation grid 
token_concept_grid(
    conv_id=i,
    activations_dir="multif/activations_topk",
    last_n_tokens=n_tokens,
    top_k_features=k_features,
    cell_width=350, 
    cell_height=250
)

concept_overlap_heatmap(
    conv_id=i,
    activations_dir="multif/activations_topk",
    last_n_tokens=n_tokens,
    top_k_features=k_features,
    width=1000, 
    height=1000,
)

# matplotlib
token_concept_grid_mpl(
    conv_id=i, 
    activations_dir="multif/activations_topk", 
    last_n_tokens=n_tokens, 
    top_k_features=k_features
)

concept_overlap_heatmap_mpl(
    conv_id=i, 
    activations_dir="multif/activations_topk", 
    last_n_tokens=n_tokens, 
    top_k_features=k_features
)


  saved -> multif/plots/token_concept_grid_1005-3-en_last25_top50.html
  saved -> multif/plots/concept_overlap_1005-3-en_last25_top50.html
  saved -> multif/plots/token_concept_grid_overview_1005-3-en_last25_top50.png
  saved -> multif/plots/concept_overlap_1005-3-en_last25_top50.png


'multif/plots/concept_overlap_1005-3-en_last25_top50.png'

In [ ]:
# stats
compute_stats(da, vec_source="user")

# raw activation bar chart
feature_bar_chart(da, pattern="forgetting", vec_key="user_mean", anchor_turn=0, start=0, end=20)

# analyze
results = da.analyze(top_k=50, min_examples=5)


In [ ]:
# diff bar chart — single source
diff_bar_chart(da, results, pattern="forgetting", source="aggregate", chart_type="drop", start=0, end=20)

# diff bar chart — all sources
diff_bar_chart(da, results, pattern="interference", chart_type="drop", start=0, end=20)

# line grid
diff_line_grid(da, results, pattern="forgetting", chart_type="drop")
